# Monte Carlo Tree Search for LLM Reasoning

MCTS (Monte Carlo Tree Search) is a principled search algorithm that uses random sampling to evaluate moves. Combined with an LLM policy, it enables systematic exploration of reasoning paths with provably better exploration-exploitation balance than greedy search.

## MCTS Background

MCTS was developed for game-playing AI (Go, Chess) and consists of four phases:
1. **Selection**: traverse the tree using UCB1 (Upper Confidence Bound) to balance exploration vs exploitation
2. **Expansion**: add a new child node from the selected node
3. **Simulation (rollout)**: play out a random game from the new node to get a terminal reward
4. **Backpropagation**: update value estimates along the path from leaf to root

For LLM reasoning, the 'game' is the reasoning process:
- States: partial reasoning chains
- Actions: next reasoning step
- Policy: LLM generates candidate steps
- Value: LLM or PRM estimates solution quality
- Terminal: complete answer reached

## AlphaProof and Formal Mathematics

DeepMind's AlphaProof (2024) used MCTS with a language model policy to solve International Mathematical Olympiad problems. The key insight: formal proof systems (Lean 4) provide ground-truth verification at each step — eliminating the need for a learned value function.

Steps that compile in Lean are valid; steps that don't compile are pruned. This gives MCTS a reliable oracle that CoT and ToT lack for most tasks. AlphaProof solved 4 of 6 IMO 2024 problems, including the hardest one.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Callable
import math, random

@dataclass
class MCTSNode:
    state: str
    depth: int = 0
    visits: int = 0
    total_value: float = 0.0
    parent: Optional['MCTSNode'] = None
    children: list = field(default_factory=list)
    terminal: bool = False
    answer: str = ''

    @property
    def q_value(self) -> float:
        return self.total_value / self.visits if self.visits > 0 else 0.0

    def ucb1(self, c: float = 1.41) -> float:
        if self.visits == 0:
            return float('inf')
        parent_visits = self.parent.visits if self.parent else self.visits
        return self.q_value + c * math.sqrt(math.log(parent_visits) / self.visits)

class LLMMCTSSolver:
    def __init__(self, policy_fn: Callable, value_fn: Callable,
                 branch_factor: int = 3, max_depth: int = 5,
                 n_simulations: int = 20, c: float = 1.41):
        self.policy = policy_fn
        self.value = value_fn
        self.branch_factor = branch_factor
        self.max_depth = max_depth
        self.n_simulations = n_simulations
        self.c = c

    def select(self, node: MCTSNode) -> MCTSNode:
        while node.children and not node.terminal:
            node = max(node.children, key=lambda n: n.ucb1(self.c))
        return node

    def expand(self, node: MCTSNode) -> MCTSNode:
        if node.terminal or node.depth >= self.max_depth:
            return node
        thoughts = self.policy(node.state, self.branch_factor)
        for thought in thoughts:
            child = MCTSNode(state=thought, depth=node.depth+1, parent=node)
            node.children.append(child)
        return node.children[0] if node.children else node

    def rollout(self, node: MCTSNode) -> float:
        return self.value(node.state, node.depth)

    def backprop(self, node: MCTSNode, value: float):
        while node:
            node.visits += 1
            node.total_value += value
            node = node.parent

    def solve(self, problem: str) -> dict:
        root = MCTSNode(state=problem)
        for _ in range(self.n_simulations):
            leaf = self.select(root)
            if leaf.visits > 0 and not leaf.terminal:
                leaf = self.expand(leaf)
            value = self.rollout(leaf)
            self.backprop(leaf, value)
        # Best path: follow highest Q-value children
        path = [root.state]
        node = root
        while node.children:
            node = max(node.children, key=lambda n: n.q_value)
            path.append(node.state)
        total_nodes = self._count_nodes(root)
        return {'path': path, 'best_q': root.q_value, 'total_nodes': total_nodes,
                'simulations': self.n_simulations}

    def _count_nodes(self, node: MCTSNode) -> int:
        return 1 + sum(self._count_nodes(c) for c in node.children)

rng = random.Random(42)

def llm_policy(state: str, k: int) -> list:
    steps = [
        f'Decompose: identify variables and constraints in: {state[:40]}',
        f'Compute intermediate result using key relationship',
        f'Verify partial solution against problem constraints',
        f'Apply relevant formula or identity to current state',
        f'Simplify: reduce complexity of current expression',
    ]
    return rng.sample(steps, min(k, len(steps)))

def llm_value(state: str, depth: int) -> float:
    return min(1.0, 0.3 + depth * 0.1 + len(state) / 500 + rng.gauss(0, 0.1))

solver = LLMMCTSSolver(llm_policy, llm_value, branch_factor=3, max_depth=4, n_simulations=15)
result = solver.solve('How many integers from 1 to 100 are divisible by both 3 and 7?')
print(f'MCTS Results:')
print(f'  Total nodes explored: {result["total_nodes"]}')
print(f'  Simulations run: {result["simulations"]}')
print(f'  Best path Q-value: {result["best_q"]:.3f}')
print(f'  Reasoning path ({len(result["path"])} steps):')
for i, step in enumerate(result['path']):
    print(f'    Step {i}: {step[:70]}')

## MCTS vs ToT vs Self-Consistency

| Property | Self-Consistency | Tree of Thought | MCTS |
|----------|-----------------|-----------------|------|
| Search strategy | Parallel sampling | BFS/DFS | UCB-guided |
| Backtracking | No | Yes (DFS) | Yes |
| Value model | None | LLM evaluator | LLM/PRM value fn |
| Exploration-exploitation | Random | Score-guided | Principled UCB |
| Cost | K × base | K^D × base | N_sim × base |
| Best for | Independent samples | Structured search | Sequential decisions |

MCTS is theoretically superior for problems with sequential dependencies but is rarely the right choice in practice due to cost. Self-consistency with K=10-20 outperforms MCTS on most standard benchmarks while being much simpler.

## When MCTS Makes Sense

MCTS is justified when:
1. **A reliable value function exists** — formal verification (code execution, proof checking) provides ground truth at each step
2. **The problem has high branching factor** — many possible next steps, hard to evaluate without exploring
3. **Latency is not a constraint** — batch reasoning where quality matters more than speed
4. **Tasks are sequential and cumulative** — early mistakes cascade (theorem proving, multi-step coding)

For most LLM applications, self-consistency over CoT is the better default. Reserve MCTS for formal domains where you have a verifier.